This notebook performs webscraping to loop over the hrefs scraped previously and scrapes the user reviews on the product page.

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import numpy as np
import re
from datetime import datetime
import time

In [2]:
options = Options()

#options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-infobars')
options.add_argument('--disable-extensions')
options.add_argument('--disable-notifications')
options.add_argument('--window-size=1724,968')
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.110 Safari/537.36"
options.add_argument(f'user-agent={user_agent}')
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [3]:
# load product href csvs
red_href_df = pd.read_csv("../data/Red_wine_hrefs.csv")
white_href_df = pd.read_csv("../data/White_wine_hrefs.csv")
sparkling_href_df = pd.read_csv("../data/Champagne-Sparkling_wine_hrefs.csv")

# Combine the DataFrames into a single DataFrame
href_df = pd.concat([red_href_df, white_href_df, sparkling_href_df], ignore_index=True)

In [4]:
href_df

,href,type
0,https://www.danmurphys.com.au/product/DM_17275...,Red
1,https://www.danmurphys.com.au/product/DM_10535...,Red
2,https://www.danmurphys.com.au/product/DM_90074...,Red
3,https://www.danmurphys.com.au/product/DM_90104...,Red
4,https://www.danmurphys.com.au/product/DM_14446...,Red
...,...,...
6542,https://www.danmurphys.com.au/product/DM_13411...,Champagne-Sparkling
6543,https://www.danmurphys.com.au/product/DM_12199...,Champagne-Sparkling
6544,https://www.danmurphys.com.au/product/DM_10851...,Champagne-Sparkling
6545,https://www.danmurphys.com.au/product/DM_10750...,Champagne-Sparkling


## Webscraping
The following cell will perform scraping over the hrefs. The scraping takes a long time and if it fails before completing, the remaining cells of the notebook can be run to process and save the data that has been scraped. Set the starting_i variable in the following cell to the index of the href from which you would like to start scraping. This allows the process to be started where the previous run failed.
## The features to be scraped are:
### username
### rating
### date

In [5]:
%%time

# exception used to cancel scraping when progression can't continue
class NoProgressionException(Exception):
    pass

hrefs_subset = []
all_reviews = []

total_wines = len(href_df)
starting_i = 0
try:
    for i in range(starting_i, len(href_df), 1):
        product_reviews = []
        
        # get href
        href = href_df.iloc[i]["href"]
        print("getting product page for", href)
        driver.get(href)
    
    
        product_exists = False
        try:
            review_cards_block = driver.find_element(By.CLASS_NAME, "not-found__content")
            print("No product page for i =", i, "-- href +", href)
        except NoSuchElementException:
            product_exists = True
    
        # check if product page exists
        if product_exists:
            is_reviewed = False
            try:
                review_cards_block = driver.find_element(By.CLASS_NAME, "customer-reviews__cards")
                is_reviewed = True
            except NoSuchElementException:
                print("!!! no product reviews for i =", i, "-- href =", href)
        
            # check if product has reviews
            if is_reviewed:
                print("reviews found")
                # loop through review pages
                while True:
                    print("starting while loop through review pages...")
                    # scraping
                    ## get review cards and loop through them
                    review_cards_block = driver.find_element(By.CLASS_NAME, "customer-reviews__cards") # may not need
                    review_cards = review_cards_block.find_elements(By.CLASS_NAME, "customer-reviews__card")
                    for review_card in review_cards:
                        print("scraping review card...")
                        try:
                            ## scrape username and href
                            user_link = review_card.find_element(By.CLASS_NAME, "review-card__sdesc-link")
                            user_href = user_link.get_attribute("href")
                            user_name = user_link.text
    
                        except NoSuchElementException:
                            ## scrape username without href
                            name_block = review_card.find_element(By.CLASS_NAME, "review-card__sdesc")
                            user_name = name_block.find_element(By.CLASS_NAME, "ng-star-inserted").text
                            #user_href = np.nan
                            user_href = "NaN"
    
                        # scrape rating
                        rating_icons = review_card.find_elements(By.CLASS_NAME, "rating-icon")
                        rating = sum(1 for icon in rating_icons if "checked" in icon.get_attribute("class"))
    
                        # scrape date
                        rating_date = review_card.find_element(By.CLASS_NAME, "review-card__info").text
                        date_pattern = r"\b(\d{1,2}\s+\w+\s+\d{4})\b"
                        match = re.search(date_pattern, rating_date)
                        extracted_date = datetime.strptime(match.group(1), "%d %B %Y").strftime("%d/%m/%Y")
    
                        # build dict for review card
                        user_review = {
                            "href" : user_href,
                            "username" : user_name,
                            "rating" : rating,
                            "date" : extracted_date
                        }
                        product_reviews.append(user_review)
    
                    # handle next reviews button
                    try:
                        button = driver.find_element(By.XPATH, "//button[@aria-label='Next page button']")
                        
                        if button.is_enabled():
    
                            # handle waiting until review cards updated
                            previous_review_block_text = review_cards_block.text
                            
                            # Click the button
                            button.click()
                            print("Next button clicked!")
                            while_loop_limit = 100
                            while_loop_counter = 0
                            reclick_limit = 3
                            reclick_counter = 0
                            while True:
                                current_review_block_text = review_cards_block.text
                                if current_review_block_text != previous_review_block_text:
                                    break
                                if while_loop_counter < while_loop_limit:
                                    time.sleep(0.1)
                                    print("waiting for reviews to update...")
                                else:
                                    if reclick_counter < reclick_limit:
                                        print("!!! reviews have not updated. Re-pressing next button")
                                        button.click()
                                        reclick_counter += 1
                                        while_loop_counter = 0
                                        time.sleep(0.5)
                                    else:
                                        print("!!! exceeded reclicks. cancelling scraping")
                                        raise NoProgressionException("Error progressing in the scraping")
                                        

                                while_loop_counter += 1
                                    
                                
                        else:
                            print("Next button is disabled. No more review pages.")
                            break
    
                    
                    except NoSuchElementException:
                        print("Couldn't find next button. No more review pages")
                        break
    
    
    
                all_reviews.append(product_reviews)
    
            
            else:
                all_reviews.append([])
            
        else:
            all_reviews.append([])
    
        hrefs_subset.append(href)
        print("scraped reviews from i =", i, "of", total_wines-1, "\n")
except Exception as e:
    print("Stopped scraping at i =", i)
    print(e)
    

getting product page for https://www.danmurphys.com.au/product/DM_172754/penfolds-bin-128-shiraz-2019?isFromSearch=false&isSponsored=true&pageName=category_page&promoteIQCampaignID=277455
reviews found
starting while loop through review pages...
scraping review card...
scraping review card...
Couldn't find next button. No more review pages
scraped reviews from i = 0 of 6546 

getting product page for https://www.danmurphys.com.au/product/DM_105353/penfolds-bin-150-marananga-shiraz-2018?isFromSearch=false&isSponsored=true&pageName=category_page&promoteIQCampaignID=277455
!!! no product reviews for i = 1 -- href = https://www.danmurphys.com.au/product/DM_105353/penfolds-bin-150-marananga-shiraz-2018?isFromSearch=false&isSponsored=true&pageName=category_page&promoteIQCampaignID=277455
scraped reviews from i = 1 of 6546 

getting product page for https://www.danmurphys.com.au/product/DM_900745/frescobaldi-castiglioni-chianti?isFromSearch=false&isSponsored=true&pageName=category_page&promot

In [9]:
user_reviews_df = {
    "href" : hrefs_subset,
    "all_reviews" : all_reviews
}

user_reviews_df = pd.DataFrame(user_reviews_df)

In [10]:
user_reviews_df

,href,all_reviews
0,https://www.danmurphys.com.au/product/DM_17275...,"[{'href': 'NaN', 'username': 'Mr Mooga', 'rati..."
1,https://www.danmurphys.com.au/product/DM_10535...,[]


In [12]:
# save df to csv
file_path = f"../data/all_user_review_subsets/user_reviews_dataset_i_{starting_i}-{i-1}.csv"
user_reviews_df.to_csv(file_path, index=False)